# LLM을 활용한 Vibe Coding Level Up

**목표:** LLM의 기본 사용법을 익히고, API를 통해 AI를 서비스와 데이터에 연결하는 흐름을 직접 실습합니다.

> 실습 흐름: **LLM 이해 → API 호출 → 스트리밍 → 대화 연속성 → 구조화된 데이터 → 서비스 활용**

### 가상환경 생성(터미널에서 실행)

```bash
python3 -m venv .venv
source .venv/bin/activate

pip install openai python-dotenv
```

`.env` 파일에 `OPENAI_API_KEY=여기에_발급받은_키`를 저장하세요. 키를 노트북 코드나 출력 결과에 직접 입력하지 마세요.

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 또는 환경변수에 OPENAI_API_KEY를 설정하세요.")

print("API 키가 설정되었습니다.")

API 키가 설정되었습니다.


## OpenAI Developers 문서

[OpenAI 개발자 문서 바로가기](https://developers.openai.com/api/docs)

In [12]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-6-luna"  # 계정에서 사용할 수 있는 모델로 변경하세요
print("OpenAI client 준비 완료")

OpenAI client 준비 완료


## Responses API

이 노트북에서는 `client.responses.create()`를 이용해 기본 호출부터 스트리밍, 대화 연속성, Structured Outputs까지 실습합니다.

- `model`: 사용할 모델
- `instructions`: 모델의 역할과 응답 규칙
- `input`: 사용자의 질문 또는 작업 지시
- `response.output_text`: 모델이 생성한 텍스트
- `stream=True`: 생성되는 텍스트를 순차적으로 받기
- `previous_response_id`: 이전 응답을 다음 요청의 문맥으로 연결하기

In [13]:
response = client.responses.create(
    model=MODEL,
    input="Write a short bedtime story about a unicorn. 한글로 작성해줘",
)

print(response.output_text)

AuthenticationError: Error code: 401 - {'error': {'message': 'Your API key has been invalidated.', 'type': None, 'code': 'token_invalidated', 'param': None}, 'status': 401}

In [5]:
response = client.responses.create(
    model=MODEL,
    tools=[{"type": "web_search"}],
    input="오늘의 주요 뉴스를 한 가지 알려주세요.",
)

print(response.output_text)

**오늘의 주요 뉴스(9월 28일):** 트럼프 미국 대통령이 이란의 호르무즈 해협 재개방 제안을 거부한 뒤 국제유가가 1% 넘게 반등했습니다. 양측의 입장 차가 이어지면서 중동 정세와 원유 공급에 대한 불안이 커지고 있습니다. ([marketscreener.com](https://www.marketscreener.com/news/oil-heads-higher-as-us-iran-peace-talks-in-stalemate-ce785adcd880f024?utm_source=openai))


## 스트리밍(Streaming)

`stream=True`를 지정하면 `response.output_text.delta` 이벤트를 순회할 수 있습니다.

In [6]:
stream = client.responses.create(
    model=MODEL,
    instructions="당신은 파이썬 프로그래머입니다.",
    input="피보나치 수열을 생성하는 파이썬 프로그램을 작성해주세요.",
    stream=True,
)

final_answer = []
for event in stream:
    if event.type == "response.output_text.delta":
        final_answer.append(event.delta)
        print(event.delta, end="")

final_answer = "".join(final_answer)

다음은 입력한 개수만큼 피보나치 수열을 출력하는 파이썬 프로그램입니다. 수열은 `0, 1`부터 시작합니다.

```python
n = int(input("몇 개의 항을 출력할까요? "))

if n < 0:
    print("0 이상의 수를 입력해주세요.")
else:
    a, b = 0, 1

    for _ in range(n):
        print(a, end=" ")
        a, b = b, a + b
```

예를 들어 `7`을 입력하면 `0 1 1 2 3 5 8`이 출력됩니다.

In [8]:
print(stream)

## 연쇄적인 대화

`previous_response_id`로 이전 응답을 다음 요청의 문맥으로 전달할 수 있습니다. `instructions`는 요청마다 전달해야 합니다.

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions="You are a helpful assistant. You must answer in Korean.",
    input="대한민국의 수도는 어디인가요?",
)

print(response.output_text)

In [ ]:
def ask(question, previous_response_id=None, model=MODEL):
    return client.responses.create(
        model=model,
        instructions="당신은 친절한 AI 튜터입니다.",
        input=question,
        previous_response_id=previous_response_id,
    )

first_response = ask("양자역학에 대해서 쉽게 설명해 주세요")
print(first_response.output_text)

In [ ]:
second_response = ask(
    "이전의 내용을 영어로 답변해 주세요",
    previous_response_id=first_response.id,
)
print(second_response.output_text)

## Structured Outputs

`text.format`에 JSON Schema를 지정하면 프로그램에서 바로 사용할 수 있는 구조화된 데이터를 받을 수 있습니다.

In [9]:
capital_schema = {
    "type": "object",
    "properties": {"capital": {"type": "string"}},
    "required": ["capital"],
    "additionalProperties": False,
}

capital_response = client.responses.create(
    model=MODEL,
    instructions="You are a helpful assistant. You must answer in Korean.",
    input="대한민국의 수도는 어디인가요?",
    text={
        "format": {
            "type": "json_schema",
            "name": "country_capital",
            "strict": True,
            "schema": capital_schema,
        }
    },
)

print(capital_response.output_text)

{"capital":"서울특별시"}


In [10]:
quiz_schema = {
    "type": "object",
    "properties": {
        "question": {"type": "string"},
        "choices": {"type": "array", "items": {"type": "string"}},
        "answer_index": {"type": "integer"},
        "difficulty": {"type": "string", "enum": ["상", "중", "하"]},
    },
    "required": ["question", "choices", "answer_index", "difficulty"],
    "additionalProperties": False,
}

quiz_response = client.responses.create(
    model=MODEL,
    input="통계를 주제로 4지선다형 객관식 문제를 하나 만들어주세요.",
    text={"format": {"type": "json_schema", "name": "statistics_quiz", "strict": True, "schema": quiz_schema}},
)
print(quiz_response.output_text)

{"question":"자료 2, 4, 6, 8의 평균은 얼마인가요?","choices":["4","5","6","8"],"answer_index":1,"difficulty":"하"}


In [11]:
import json
quiz_obj = json.loads(quiz_response.output_text)
quiz_obj

{'question': '자료 2, 4, 6, 8의 평균은 얼마인가요?',
 'choices': ['4', '5', '6', '8'],
 'answer_index': 1,
 'difficulty': '하'}

# 음성 명령과 Function Calling

## 음성 녹음과 OpenAI 음성 전사(STT)

설치: `pip install numpy scipy sounddevice`

완료된 WAV 파일은 `client.audio.transcriptions.create()`와 `gpt-transcribe`로 전사합니다.

In [12]:
%pip install numpy scipy sounddevice

  Using cached numpy-2.5.3-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached cffi-2.1.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.5 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached numpy-2.5.3-cp313-cp313-macosx_14_0_arm64.whl (5.4 MB)
Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl (20.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
Using cached cffi-2.1.1-cp313-cp313-macosx_11_0_arm64.whl (184 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sounddevice] [scipy]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import queue
import sys
import numpy as np
import sounddevice as sd
from scipy.io import wavfile

def record_user_voice(filename="my_voice.wav", fs=44100):
    print("🎤 말씀해 주세요. 녹음을 끝내려면 Enter를 누르세요.")

    audio_queue = queue.Queue()

    def callback(indata, frames, time, status):
        if status:
            print(status, file=sys.stderr)
        audio_queue.put(indata.copy())

    with sd.InputStream(
        samplerate=fs,
        channels=1,
        callback=callback
    ):
        input()

    chunks = []
    while not audio_queue.empty():
        chunks.append(audio_queue.get())

    if not chunks:
        raise RuntimeError("녹음된 오디오가 없습니다.")

    recording = np.concatenate(chunks, axis=0)
    wavfile.write(filename, fs, recording)
    print(f"✅ 저장 완료: {filename}")


def transcribe_voice(filename="my_voice.wav"):
    with open(filename, "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="gpt-transcribe",
            file=audio_file,
        )

    print("📝 전사 결과:", transcription.text)
    return transcription.text.strip()


record_user_voice()
FINAL_TRANSCRIPTION = transcribe_voice()

🎤 말씀해 주세요. 녹음을 끝내려면 Enter를 누르세요.
✅ 저장 완료: my_voice.wav
📝 전사 결과: 안녕.


## Responses API Function Calling

Responses API에서는 함수 도구의 `name`, `description`, `parameters`를 도구 객체에 직접 지정합니다. `response.output`의 `function_call`을 실행한 뒤, `function_call_output`을 다음 요청의 `input`으로 전달합니다.

In [21]:
import json

def open_refrigerator_door(door):
    return {"success": True, "door": door, "message": "냉장고 문을 열었습니다. (실습용 시뮬레이션)"}

tools = [{
    "type": "function",
    "name": "open_refrigerator_door",
    "description": "사용자가 냉장고 문을 열어 달라고 요청하면 냉장고 문을 엽니다.",
    "parameters": {
        "type": "object",
        "properties": {"door": {"type": "string", "enum": ["main"]}},
        "required": ["door"],
        "additionalProperties": False,
    },
    "strict": True,
}]

prompt = globals().get("FINAL_TRANSCRIPTION", "").strip() or "냉장고 문 열어줘"
print("사용자 명령:", prompt)

response = client.responses.create(
    model=MODEL,
    instructions="당신은 냉장고 제어 도우미입니다. 필요한 경우 제공된 함수만 호출하세요.",
    input=prompt,
    tools=tools,
)

tool_outputs = []
for item in response.output:
    if item.type != "function_call":
        continue
    if item.name != "open_refrigerator_door":
        raise ValueError(f"허용되지 않은 함수입니다: {item.name}")
    arguments = json.loads(item.arguments)
    result = open_refrigerator_door(**arguments)
    print("Function Call:", item.name, arguments)
    print("함수 실행 결과:", result)
    tool_outputs.append({
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": json.dumps(result, ensure_ascii=False),
    })

if tool_outputs:
    final_response = client.responses.create(
        model=MODEL,
        instructions="실행 결과를 바탕으로 사용자에게 한국어로 짧게 안내하세요.",
        previous_response_id=response.id,
        input=tool_outputs,
        tools=tools,
    )
    print("인공지능 최종 답변:", final_response.output_text)
else:
    print("일반 답변:", response.output_text)

사용자 명령: 안녕.
일반 답변: 안녕하세요! 무엇을 도와드릴까요?


# 🤖 AI 에이전트 핵심 기술 4대 요소

---

### 📊 핵심 요소 요약 비교표

| 기술 요소 | 💡 핵심 개념 | 🏗️ 개발자 관점 (역할과 가치) | 👥 일반 사용자 관점 (체감 효과) |
| :--- | :--- | :--- | :--- |
| **시스템 프롬프트**<br>*(System Prompt)* | AI에게 부여하는 **기본 성격과 행동 규칙** | 답변 톤앤매너 강제, 보안 취약점 방지 1차 방어 | AI의 전문성(예: 번역가 등) 패르소나 | 
| **펑션 콜**<br>*(Function Calling)* | LLM이 외부 시스템을 잇는 **동작 인터페이스** | 자연어에서 필요한 파라미터(함수명)를 추출 후 실행 | "말만 했는데 진짜로 되네?" 실질적 기능 체감 |
| **MCP**<br>*(Model Context Protocol)* | 인터넷상 데이터 소스를 연결하는 **통합 표준 규격** | 플러그인 방식의 다양한 데이터 즉시 공급 | 슬랙 대화, 노션 문서 등을 AI가 연동 처리 | 
| **하네스 엔지니어링**<br>*(Harness Engineering)* | AI가 안전하게 일할 수 있는 **환경 및 제어 설계** | 자동화, 루프, 격리된 가상 공간 제공 | 할루시네이션 및 AI 보안 취약점 등 개선  | 

---